# 01. Estructura del Mercado Español y Restricciones BME
**Proyecto:** Framework Cuantitativo — Concurso de Bolsa BME 2026  
**Objetivo:** Analizar la microestructura del mercado bursátil español (IBEX 35), identificar los límites de liquidez y modelar matemáticamente el impacto de las restricciones y costes del concurso (canon mínimo de 7 € y asimetría de la puntuación oficial).


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración estética
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Entorno de análisis inicializado con éxito.")


## 1. Fotografía del Mercado: Concentración y Oligopolio Sectorial
El mercado continuo español presenta una estructura altamente concentrada. A diferencia de índices diversificados como el S&P 500, el IBEX 35 está dominado por un puñado de grandes conglomerados financieros y de infraestructuras/utilities.


In [ ]:
# Carga de metadatos de composición de mercado
df_market = pd.read_csv('../data/raw/market_composition.csv')
print(f"Total empresas en el universo registrado: {len(df_market)}")
print(f"Empresas pertenecientes al IBEX 35: {df_market['Is_IBEX35'].sum()}")
df_market.head(5)


In [ ]:
# 1.1 Top 10 Empresas por Capitalización Bursátil
top_10 = df_market.sort_values(by='MarketCap_B', ascending=False).head(10)

plt.figure(figsize=(12, 5))
bars = plt.barh(top_10['Name'], top_10['MarketCap_B'], color='#1f77b4', edgecolor='black')
plt.xlabel('Capitalización Bursátil (Miles de Millones de €)')
plt.title('Top 10 Empresas por Capitalización en España')
plt.gca().invert_yaxis()

for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.8, bar.get_y() + bar.get_height()/2, f'€{width:.1f}B', 
             va='center', ha='left', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# 1.2 Distribución Sectorial del Mercado Español
sector_mc = df_market.groupby('Sector')['MarketCap_B'].sum().sort_values(ascending=False)
colors = sns.color_palette('Blues_r', len(sector_mc))

plt.figure(figsize=(10, 6))
plt.pie(sector_mc, labels=sector_mc.index, autopct='%1.1f%%', startangle=140, 
        colors=colors, explode=[0.05 if i < 2 else 0 for i in range(len(sector_mc))],
        wedgeprops={'edgecolor': 'black', 'linewidth': 1})
plt.title('Concentración Sectorial por Capitalización Bursátil en España')
plt.tight_layout()
plt.show()

print("Resumen de Concentración:")
for sector, mc in sector_mc.items():
    print(f" - {sector}: €{mc:.2f}B ({mc / sector_mc.sum() * 100:.1f}%)")


> [!IMPORTANT]
> **Hallazgo Estructural 1:** Más del **60% de la capitalización** del mercado español se concentra en apenas dos sectores: **Servicios Financieros (Banca)** y **Utilities / Energía**. Diseñar una estrategia cuantitativa en España exige comprender a fondo la dinámica de estos dos sectores.


## 2. Liquidez y Filtro de Fricción Operativa
En el concurso de BME, operar activos de baja liquidez ("chicharros") acarrea dos riesgos críticos:
1. **Regla de los 2 minutos:** El simulador cancela órdenes si no cruzan en 2 minutos por falta de contrapartida.
2. **Slippage y Market Impact:** En cuentas reales, introducir órdenes de miles de euros en activos ilíquidos distorsiona los precios.

Evaluamos el **percentil 10** del volumen diario negociado en euros (`Volume * Close`) durante los últimos 15 años para medir la liquidez en el peor escenario histórico.


In [ ]:
# Cargar histórico de precios de cierre y volúmenes
prices = pd.read_csv('../data/raw/stocks_close_15y.csv', index_col=0, parse_dates=True)
volumes = pd.read_csv('../data/raw/stocks_volume_15y.csv', index_col=0, parse_dates=True)

# Volumen diario negociado en Euros
euro_volume = volumes * prices

# Percentil 10 histórico de volumen en euros por activo
p10_euro_vol = euro_volume.quantile(0.10)

# Umbral de seguridad: la orden de 10.000 € no debe superar el 1% del volumen diario del percentil 10
target_order_size = 10000
slippage_threshold_ratio = 0.01
max_safe_order = p10_euro_vol * slippage_threshold_ratio

liquidity_df = pd.DataFrame({
    'P10_Daily_Euro_Vol': p10_euro_vol,
    'Max_Safe_Order (1%)': max_safe_order
}).dropna()

unsafe_assets = liquidity_df[liquidity_df['Max_Safe_Order (1%)'] < target_order_size].sort_values('P10_Daily_Euro_Vol')
safe_assets = liquidity_df.drop(unsafe_assets.index).sort_values('P10_Daily_Euro_Vol', ascending=False)

print(f"Total activos evaluados: {len(liquidity_df)}")
print(f"Activos que superan el filtro de liquidez institucional: {len(safe_assets)}")
print(f"Activos descartados por iliquidez: {len(unsafe_assets)}")


In [ ]:
# Gráfico de distribución de liquidez (Percentil 10)
plt.figure(figsize=(14, 6))
plt.bar(safe_assets.index, safe_assets['P10_Daily_Euro_Vol'] / 1e6, color='#2ca02c', label='Apto para operativa institucional')
if len(unsafe_assets) > 0:
    plt.bar(unsafe_assets.index, unsafe_assets['P10_Daily_Euro_Vol'] / 1e6, color='#d62728', label='Descartado (Riesgo de iliquidez / ejecución)')

# Línea de corte para absorber 10.000 € al 1% de participación diaria = 1.000.000 € de volumen mínimo
req_vol_millions = (target_order_size / slippage_threshold_ratio) / 1e6
plt.axhline(req_vol_millions, color='black', linestyle='--', linewidth=2, label=f'Volumen Mínimo Seguro (€{req_vol_millions:.0f}M)')

plt.yscale('log')
plt.title('Percentil 10 de Volumen Negociado Diario (€ Millones, Escala Logarítmica)')
plt.ylabel('Volumen Diario en € (Millones)')
plt.xticks(rotation=90)
plt.legend()
plt.tight_layout()
plt.show()


## 3. Modelado Matemático de la Fricción del Bróker
Las bases de BME fijan una comisión de intermediación de:
$$\text{Comisión} = \max(7.00\text{ €}, 0.15\% \times \text{Efectivo})$$

El canon fijo de **7 €** introduce una fuerte asimetría matemática en el coste porcentual por operación:
$$\text{Coste Efectivo \%} = \frac{\max(7.00, 0.0015 \times V)}{V} \times 100$$


In [ ]:
# Curva de fricción porcentual según el volumen de la orden
order_sizes = np.linspace(200, 15000, 300)
commission_euros = np.maximum(7.0, order_sizes * 0.0015)
commission_pct = (commission_euros / order_sizes) * 100

plt.figure(figsize=(12, 5))
plt.plot(order_sizes, commission_pct, color='#d62728', linewidth=2.5, label='Coste Efectivo del Canon (%)')
plt.axvline(4667, color='black', linestyle=':', label='Umbral de dilución óptima (4.667 € -> 0.15%)')
plt.axvline(900, color='#ff7f0e', linestyle='--', linewidth=2, label='Nuestra orden en Micro-Alfa (900 € -> 0.78% por orden)')

plt.title('Impacto Porcentual de la Comisión Mínima de 7 € sobre el Tamaño de Orden')
plt.xlabel('Tamaño de Orden (€)')
plt.ylabel('Coste Porcentual por Operación (%)')
plt.ylim(0, 3.5)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Coste de una orden de 500 €:  €{max(7.0, 500*0.0015):.2f} ({max(7.0, 500*0.0015)/500*100:.2f}%)")
print(f"Coste de una orden de 900 €:  €{max(7.0, 900*0.0015):.2f} ({max(7.0, 900*0.0015)/900*100:.2f}%)")
print(f"Coste de una orden de 5.000 €: €{max(7.0, 5000*0.0015):.2f} ({max(7.0, 5000*0.0015)/5000*100:.2f}%)")


> [!WARNING]
> **Implicación Cuantitativa Crucial:** 
> Los operadores activos que realizan 30 a 50 operaciones mensuales con tamaños pequeños (1.000 € a 2.000 €) pagan entre **0.35% y 0.70% por pata** (compra + venta = hasta **1.40% de sangrado neto por trade**). 
> Esta fricción destruye cualquier ventaja predictiva intradiaria.


## 4. La Función de Puntuación de BME y el Efecto Suelo
La puntuación oficial del concurso evalúa las carteras con un máximo de **9.0 puntos**:
$$\text{Puntuación Total}_i = \text{Puntos Retorno}_i (\le 6.0) + \text{Puntos Volatilidad}_i (\le 3.0)$$

Donde:
$$\text{Puntos Retorno}_i = 6.0 \times \frac{V_{\text{final}, i}}{V_{\max}}$$
$$\text{Puntos Volatilidad}_i = 3.0 \times \left(1 - \frac{\sigma_i - \sigma_{\min}}{\sigma_{\max} - \sigma_{\min}}\right)$$


In [ ]:
# Simulación del impacto de puntuación entre perfiles de competidores
v_initial = 50000.0

profiles = [
    {'Name': 'Líder Agresivo (Day Trader)', 'Return_pct': 0.08, 'Annual_Vol': 0.25},
    {'Name': 'Seguidor de Tendencia', 'Return_pct': 0.04, 'Annual_Vol': 0.18},
    {'Name': 'Buy & Hold IBEX 35', 'Return_pct': 0.02, 'Annual_Vol': 0.14},
    {'Name': 'Competidor Promedio', 'Return_pct': -0.005, 'Annual_Vol': 0.12},
    {'Name': 'Micro-Alfa Quirúrgico (Nuestro)', 'Return_pct': -0.0024, 'Annual_Vol': 0.0020}
]

df_profiles = pd.DataFrame(profiles)
df_profiles['V_final'] = v_initial * (1 + df_profiles['Return_pct'])

v_max = df_profiles['V_final'].max()
vol_min = df_profiles['Annual_Vol'].min()
vol_max = df_profiles['Annual_Vol'].max()

df_profiles['Score_Return'] = 6.0 * (df_profiles['V_final'] / v_max)
df_profiles['Score_Vol'] = 3.0 * (1 - (df_profiles['Annual_Vol'] - vol_min) / (vol_max - vol_min))
df_profiles['Score_Total'] = df_profiles['Score_Return'] + df_profiles['Score_Vol']

df_profiles = df_profiles.sort_values(by='Score_Total', ascending=False).reset_index(drop=True)
df_profiles[['Name', 'Return_pct', 'Annual_Vol', 'Score_Return', 'Score_Vol', 'Score_Total']]


In [ ]:
# Gráfico de descomposición de puntuación
plt.figure(figsize=(12, 6))
bar_width = 0.55
indices = np.arange(len(df_profiles))

p1 = plt.bar(indices, df_profiles['Score_Return'], bar_width, label='Puntos Retorno (máx 6.0)', color='#1f77b4')
p2 = plt.bar(indices, df_profiles['Score_Vol'], bar_width, bottom=df_profiles['Score_Return'], label='Puntos Volatilidad (máx 3.0)', color='#2ca02c')

plt.xticks(indices, df_profiles['Name'], rotation=15, ha='right')
plt.ylabel('Puntos Oficiales BME (sobre 9.0)')
plt.title('Simulación de Puntuación BME: La Ventaja Asimétrica de la Volatilidad Mínima')
plt.ylim(0, 10)
plt.axhline(9.0, color='gray', linestyle=':')

for i, total in enumerate(df_profiles['Score_Total']):
    plt.text(i, total + 0.15, f"{total:.2f} pts", ha='center', fontweight='bold', fontsize=11)

plt.legend()
plt.tight_layout()
plt.show()


## 5. Conclusiones y Tesis Cuantitativa del Proyecto

1. **El Mercado Español es un Oligopolio:** La concentración en Banca y Utilities exige evitar el ruido en pequeñas empresas ilíquidas.
2. **Las Comisiones Penalizan la Frecuencia:** El canon mínimo de 7 € impone un enfoque quirúrgico de mínimas operaciones obligatorias (exactamente 3 compras y 3 ventas por semana).
3. **El "Efecto Suelo" de Rentabilidad:** Con 50.000 € de capital inicial, la diferencia en puntos de retorno frente al líder del torneo es de apenas **~0.46 puntos**, mientras que la ventaja en volatilidad es de **+3.00 puntos íntegros**.
4. **Objetivo del Siguiente Notebook:** Encontrar la anomalía temporal que permita mantener la volatilidad en $\sigma \approx 0.20\%$ y seleccionar el vehículo óptimo de mínima varianza (las 5 utilities reguladas).
